In [101]:
############################################################################################################################
##Protected Planet Global Coverage Statistics: Post-processing script/ Version 1.0. Written by: Lewis Kramer (28/09/2025).##
############################################################################################################################

#Import requried packages
import pandas as pd
from datetime import datetime

# Get current month and year (e.g. "Sep2025") to print on the output .csv
timestamp = datetime.now().strftime("%b%Y")

Sep2025


In [107]:
# Step 1. 
# Preprocess global coverage statistics output (new model). Replace all spaces and protected characters, converting cell values to float. 

# Download csvs to local drive (new model outputs) from Metabase. Enquire into pulling data in via API.  
global_stats_all_abnj = pd.read_csv(
    r"C:\Users\lewisk\OneDrive - WCMC\Documents\UNEP-WCMC\WDPA\Global_Stats_Python_Script\original_data\sep2025__global_stats__iso3_all__and_abnj_only_2025-09-29.csv"
)

# List fields with numeric values, to be changed to float, and cleaned. 
numeric_cols = ["WDPA Mar.", "WDPA Ter.", "OECM Mar.", "OECM Ter.", "WDPA & OECM Mar.", "WDPA & OECM Ter."]

# Clean field headers by removing trailing space, replace spaces with underscores, remove periods, remove unique characters e.g., &, convert all to lowercase. 
global_stats_all_abnj.columns = (
    global_stats_all_abnj.columns
    .str.strip()           
    .str.replace(" ", "_") 
    .str.replace(".", "", regex=False) 
    .str.replace("&", "and", regex=False) 
    .str.lower()           
)

# List of numeric columns (after cleaning)
numeric_cols = ["wdpa_mar", "wdpa_ter", "oecm_mar", "oecm_ter", "wdpa_and__oecm_mar", "wdpa_and_oecm_ter"]

# Convert numeric columns to float. Clean data at same time, removing spaces in numbers, and replacing commas with "."
for col in numeric_cols:
    global_stats_all_abnj[col] = (
        global_stats_all_abnj[col]
        .astype(str)
        .str.replace(" ", "")  
        .str.replace(",", ".")  
        .astype(float)
    )

# Print cleaned dataset to check how it looks
global_stats_all_abnj.head()


,unique_process_id,iso3,wdpa_mar,wdpa_ter,oecm_mar,oecm_ter,wdpa_and__oecm_mar,wdpa_and_oecm_ter
0,_GS_20250913-1744_newC_grid_1x1dd_Sep2025_all-...,ALL,34882837.0,22100532.0,860322.0,1481782.0,35743158.0,23582314.0
1,_NS_20250813-1644_newC_grid_1x1dd_Sep2025_all-...,ABNJ,3172810.0,48889.0,462863.0,0.0,3635674.0,48889.0


In [114]:
# Step 2. 
# Isolate ABNJ values for calculation of global coverage stats. ABNJ coverage is based on outputs from the national stats process. 

# Select ALL and ABNJ rows from dataframe, and store seperately. 
global_coverage_all_modeloutput = global_stats_all_abnj[global_stats_all_abnj["iso3"] == "ALL"]
global_coverage_abnj_modeloutput = global_stats_all_abnj[global_stats_all_abnj["iso3"] == "ABNJ"]

# Subtract ABNJ values from ALL values for numeric columns
diff_values = global_coverage_all_modeloutput[numeric_cols].values - global_coverage_abnj_modeloutput[numeric_cols].values

# Create the new row in data frame
diff_row = global_coverage_all_modeloutput.copy()
diff_row.loc[:, numeric_cols] = diff_values
diff_row.loc[:, "iso3"] = "ALL_minus_ABNJ"

# Remove any existing "ALL_minus_ABNJ" row (prevents duplication)
global_stats_all_abnj = global_stats_all_abnj[global_stats_all_abnj["iso3"] != "ALL_minus_ABNJ"]

# Append the new version
global_stats_all_abnj = pd.concat([global_stats_all_abnj, diff_row], ignore_index=True)

# Review outputs
print(global_stats_all_abnj.tail())


                                   unique_process_id            iso3  \
0  _GS_20250913-1744_newC_grid_1x1dd_Sep2025_all-...             ALL   
1  _NS_20250813-1644_newC_grid_1x1dd_Sep2025_all-...            ABNJ   
2  _GS_20250913-1744_newC_grid_1x1dd_Sep2025_all-...  ALL_minus_ABNJ   

     wdpa_mar    wdpa_ter  oecm_mar   oecm_ter  wdpa_and__oecm_mar  \
0  34882837.0  22100532.0  860322.0  1481782.0          35743158.0   
1   3172810.0     48889.0  462863.0        0.0           3635674.0   
2  31710027.0  22051643.0  397459.0  1481782.0          32107484.0   

   wdpa_and_oecm_ter  
0         23582314.0  
1            48889.0  
2         23533425.0  


In [122]:
# Step 3.
# Calculate PA ABNJ area and OECM ABNJ area. ABNJ area should only be marine area, so land area added to marine area to produce single PA ABNJ (marine) area value and single OECM ABNJ (marine) area. 

# Filter ABNJ row
abnj_data = global_stats_all_abnj[global_stats_all_abnj["iso3"] == "ABNJ"]

# Calculate combined abnj protected area coverage (abnj marine + abnj terrestrial), abnj OECM coverage, and abnj protected area and OECM coverage
abnj_pa_total = abnj_data["wdpa_mar"].sum() + abnj_data["wdpa_ter"].sum()
abnj_oecm_total = abnj_data["oecm_mar"].sum() + abnj_data["oecm_ter"].sum()
abnj_pa_oecm_total = abnj_data["wdpa_and__oecm_mar"].sum() + abnj_data["wdpa_and_oecm_ter"].sum()

# Print to check values/outputs.
print("Combined ABNJ PA area (WDPA Mar + WDPA Ter):", abnj_pa_total)
print("Combined ABNJ OECM area (OECM Mar + OECM Ter):", abnj_oecm_total)
print("Combined ABNJ PA OECM area (OECM Mar + OECM Ter):", abnj_pa_oecm_total)

Combined ABNJ PA area (WDPA Mar + WDPA Ter): 3221699.0
Combined ABNJ OECM area (OECM Mar + OECM Ter): 462863.0
Combined ABNJ PA OECM area (OECM Mar + OECM Ter): 3684563.0


In [80]:
# Step 4. 
# Isolate ATA basemap land area value, to be subtracted from total basemap area when calculating proportion of land covered by PAs/OECMs/PA+OECMs.

#Read in export from basemap area calculations (by DT), including terretrial basemap area for ATA (Antarctica) only. 
ATA_land_area = pd.read_csv(
    r"C:\Users\lewisk\OneDrive - WCMC\Documents\UNEP-WCMC\WDPA\Global_Stats_Python_Script\original_data\basemap___only_ata_2025-09-29.csv"
)

# Clean field headers by removing trailing space, replace spaces with underscores, remove periods, remove unique characters e.g., &, convert all to lowercase. 
ATA_land_area.columns = (
    ATA_land_area.columns
    .str.strip()           
    .str.replace(" ", "_") 
    .str.replace(".", "", regex=False) 
    .str.replace("&", "and", regex=False) 
    .str.lower()           
)

# List of numeric columns (after cleaning)
numeric_cols = ["sum_of_area_km2"]

# Convert numeric columns to float
ATA_land_area["sum_of_area_km2"] = (
    ATA_land_area["sum_of_area_km2"]
    .astype(str)
    .str.replace(" ", "")       
    .str.replace(",", ".")       
    .str.strip()                
    .str.replace(r"[^\d.]", "", regex=True) 
    .astype(float)
)

# Calculate ATA terrestrial basemap area. 
basemap_ata_terrestrial_area = ATA_land_area["sum_of_area_km2"].sum()

12537755.81


In [123]:
# Step 5.
# Preprocess global basemap area values so that are readable. Replacing all spaces and protected characters, converting numerical values to float. 

# Read in basemap area calculations by DT (csv). 
basemap_area_all = pd.read_csv(
    r"C:\Users\lewisk\OneDrive - WCMC\Documents\UNEP-WCMC\WDPA\Global_Stats_Python_Script\original_data\basemap___total_area__abnj__eez___land__2025-09-29.csv"
)

# Clean field headers by removing trailing space, replace spaces with underscores, remove periods, remove unique characters e.g., &, convert all to lowercase. 
basemap_area_all.columns = (
    basemap_area_all.columns
    .str.strip()           
    .str.replace(" ", "_") 
    .str.replace(".", "", regex=False) 
    .str.replace("&", "and", regex=False) 
    .str.lower()           
)

# List of numeric columns (after cleaning)
numeric_cols = ["total_area"]

# Convert numeric columns to float
for col in numeric_cols:
    basemap_area_all[col] = (
        basemap_area_all[col]
        .astype(str)
        .str.replace(" ", "")  # remove spaces in numbers
        .str.replace(",", ".")  # optional: remove commas
        .astype(float)
    )

# Calculate required basemap area values (eez, abnj, terrestrial, terrestrial area with ATA, marine (eez+abnj), terrestrial+marine (including ATA), terrestrial+marine (excluding ATA). 
basemap_abnj_area = basemap_area_all[basemap_area_all["type"] == "ABNJ"]["total_area"].values[0]
basemap_eez_area = basemap_area_all[basemap_area_all["type"] == "EEZ"]["total_area"].values[0]
basemap_terrestrial_area = basemap_area_all[basemap_area_all["type"] == "Land"]["total_area"].values[0]
basemap_terrestrial_area_noATA = basemap_terrestrial_area - basemap_ata_terrestrial_area
basemap_marine_area = basemap_abnj_area + basemap_eez_area
basemap_terrestrial_marine_area = basemap_terrestrial_area + basemap_marine_area
basemap_terrestrial_marine_area_noATA = basemap_terrestrial_area_noATA + basemap_marine_area

In [133]:
#Step 6.
#Calculate area coverage (km2) and percent coverage (%).

#Select coverage values without ABNJ coverage included. ABNJ coverage to be added seperately. 
total_pa_oecm_area_model_output_with_abnj = global_stats_all_abnj[global_stats_all_abnj["iso3"] == "ALL"]
total_pa_oecm_area_model_output_minus_abnj = global_stats_all_abnj[global_stats_all_abnj["iso3"] == "ALL_minus_ABNJ"]

#Protected area coverage
total_flat_pa_area = total_pa_oecm_area_model_output_with_abnj["wdpa_mar"].sum() + total_pa_oecm_area_model_output_with_abnj["wdpa_ter"].sum()
terrestrial_pa_area = total_pa_oecm_area_model_output_minus_abnj["wdpa_ter"].sum()
eez_pa_area =  total_flat_pa_area - (terrestrial_pa_area + abnj_pa_total)
marine_pa_area = eez_pa_area + abnj_pa_total

#OECM area coverage
total_flat_oecm_area = total_pa_oecm_area_model_output_with_abnj["oecm_mar"].sum() + total_pa_oecm_area_model_output_with_abnj["oecm_ter"].sum()
terrestrial_oecm_area = total_pa_oecm_area_model_output_minus_abnj["oecm_ter"].sum()
eez_oecm_area =  total_flat_oecm_area - (terrestrial_oecm_area + abnj_oecm_total)
marine_oecm_area = eez_oecm_area + abnj_oecm_total

#Protected area and OECM coverage
terrestrial_pa_oecm_area = terrestrial_pa_area + terrestrial_oecm_area
eez_pa_oecm_area = eez_pa_area + eez_oecm_area
marine_pa_oecm_area = marine_pa_area + marine_oecm_area
abnj_pa_oecm_area = abnj_pa_total + abnj_oecm_total

#Terrestrial and marine protected area coverage
terrestrial_marine_pa_area = terrestrial_pa_area + marine_pa_area

#Terrestrial and marine OECM coverage
terrestrial_marine_oecm_area = terrestrial_oecm_area + marine_oecm_area

#Terrestrial and marine protected area and OECM coverage
terrestrial_marine_pa_oecm_area = terrestrial_pa_oecm_area + marine_pa_oecm_area

#Percent protected area coverage
terrestrial_pa_percent_coverage = (terrestrial_pa_area / basemap_terrestrial_area_noATA) * 100
eez_pa_percent_coverage = (eez_pa_area / basemap_eez_area) * 100
abnj_pa_percent_coverage = (abnj_pa_total / basemap_abnj_area) * 100
marine_pa_percent_coverage = (marine_pa_area / basemap_marine_area) * 100

#Percent OECM coverage
terrestrial_oecm_percent_coverage = (terrestrial_oecm_area / basemap_terrestrial_area_noATA) * 100
eez_oecm_percent_coverage = (eez_oecm_area / basemap_eez_area) * 100
abnj_oecm_percent_coverage = (abnj_oecm_total / basemap_abnj_area) * 100
marine_oecm_percent_coverage = (marine_oecm_area / basemap_marine_area) * 100

#Percent protected area and OECM coverage
terrestrial_pa_oecm_percent_coverage = (terrestrial_pa_oecm_area / basemap_terrestrial_area_noATA) * 100
eez_pa_oecm_percent_coverage = (eez_pa_oecm_area / basemap_eez_area) * 100
abnj_pa_oecm_percent_coverage = (abnj_pa_oecm_area / basemap_abnj_area) * 100
marine_pa_oecm_percent_coverage = (marine_pa_oecm_area / basemap_marine_area) * 100

#Percent protected area coverage (terrestrial + marine)
terrestrial_marine_pa_percent_coverage_inc_ata = (terrestrial_marine_pa_area / basemap_terrestrial_marine_area) * 100

#Percent OECM coverage (terrestrial + marine)
terrestrial_marine_oecm_percent_coverage_inc_ata = (terrestrial_marine_oecm_area / basemap_terrestrial_marine_area) * 100

#Percent protected area and OECM coverage (terrestrial + marine)
terrestrial_marine_pa_oecm_percent_coverage_inc_ata = (terrestrial_marine_pa_oecm_area / basemap_terrestrial_marine_area) * 100

#Percent protected area coverage (terrestrial + marine, no ATA in basemap area)
terrestrial_marine_pa_percent_coverage_excl_ata = (terrestrial_marine_pa_area / basemap_terrestrial_marine_area_noATA) * 100

#Percent OECM coverage (terrestrial + marine, no ATA in basemap area)
terrestrial_marine_oecm_percent_coverage_excl_ata = (terrestrial_marine_oecm_area / basemap_terrestrial_marine_area_noATA) * 100

#Percent protected area and OECM coverage (terrestrial + marine, no ATA in basemap area)
terrestrial_marine_pa_oecm_percent_coverage_excl_ata = (terrestrial_marine_pa_oecm_area / basemap_terrestrial_marine_area_noATA) * 100

In [142]:
print(terrestrial_pa_area)
print(total_flat_pa_area)
print(eez_pa_area)
print(abnj_pa_total)


22051643.0
56983369.0
31710027.0
3221699.0


In [136]:
# Step 7.
#Format outputs in table to dataframe that can be exported to .csv

#Generate dictionary/dataframe. 
summary_data = [
    {
        "type": "terrestrial",
        "basemap_area": basemap_terrestrial_area_noATA,
        "pa_area": terrestrial_pa_area,
        "pa_percent_coverage": terrestrial_pa_percent_coverage,
        "oecm_area": terrestrial_oecm_area,
        "oecm_percent_coverage": terrestrial_oecm_percent_coverage,
        "pa_oecm_area": terrestrial_pa_oecm_area,
        "pa_oecm_percent_coverage": terrestrial_pa_oecm_percent_coverage 
    },
    {
        "type": "eez",
        "basemap_area": basemap_eez_area,
        "pa_area": eez_pa_area,
        "pa_percent_coverage": eez_pa_percent_coverage,
        "oecm_area": eez_oecm_area,
        "oecm_percent_coverage": eez_oecm_percent_coverage,
        "pa_oecm_area": eez_pa_oecm_area,
        "pa_oecm_percent_coverage": eez_pa_oecm_percent_coverage,
    },
    {
        "type": "abnj",
        "basemap_area": basemap_abnj_area,
        "pa_area": abnj_pa_total,
        "pa_percent_coverage": abnj_pa_percent_coverage,
        "oecm_area": abnj_oecm_total,
        "oecm_percent_coverage": abnj_oecm_percent_coverage,
        "pa_oecm_area": abnj_pa_oecm_area,
        "pa_oecm_percent_coverage": abnj_pa_oecm_percent_coverage,
    },
    {
        "type": "marine",
        "basemap_area": basemap_marine_area,
        "pa_area": marine_pa_area,
        "pa_percent_coverage": marine_pa_percent_coverage,
        "oecm_area": marine_oecm_area,
        "oecm_percent_coverage": marine_oecm_percent_coverage,
        "pa_oecm_area": marine_pa_oecm_area,
        "pa_oecm_percent_coverage": marine_pa_oecm_percent_coverage,
    },
    {
        "type": "terrestrial+marine incl ATA",
        "basemap_area": basemap_terrestrial_marine_area,
        "pa_area": terrestrial_marine_pa_area,
        "pa_percent_coverage": terrestrial_marine_pa_percent_coverage_inc_ata,
        "oecm_area": terrestrial_marine_oecm_area,
        "oecm_percent_coverage": terrestrial_marine_oecm_percent_coverage_inc_ata,
        "pa_oecm_area": terrestrial_marine_pa_oecm_area,
        "pa_oecm_percent_coverage": terrestrial_marine_pa_oecm_percent_coverage_inc_ata,
    },
    {
        "type": "terrestrial+marine excl ATA",
        "basemap_area": basemap_terrestrial_marine_area_noATA,
        "pa_area": terrestrial_marine_pa_area,
        "pa_percent_coverage": terrestrial_marine_pa_percent_coverage_excl_ata,
        "oecm_area": terrestrial_marine_oecm_area,
        "oecm_percent_coverage": terrestrial_marine_oecm_percent_coverage_excl_ata,
        "pa_oecm_area": terrestrial_marine_pa_oecm_area,
        "pa_oecm_percent_coverage": terrestrial_marine_pa_oecm_percent_coverage_excl_ata,
    }
]

# Convert to DataFrame
summary_table = pd.DataFrame(summary_data)

# Round all fields to two decimal places
for col in ["basemap_area", "pa_area", "pa_percent_coverage", "oecm_area", "oecm_percent_coverage", "pa_oecm_area", "pa_oecm_percent_coverage"]:
    summary_table[col] = summary_table[col].round(2)

# Export table to CSV
output_path = fr"C:\Users\lewisk\OneDrive - WCMC\Documents\UNEP-WCMC\WDPA\Global_Stats_Python_Script\outputs\PP_MonthlyStats_Global_PCA_Coverage_{timestamp}.csv"

# Export to CSV
summary_table.to_csv(output_path, index=False)

# Print once export completed
print(f"Summary table exported to: {output_path}")

# Display table in Notebook
summary_table



Summary table exported to: C:\Users\lewisk\OneDrive - WCMC\Documents\UNEP-WCMC\WDPA\Global_Stats_Python_Script\outputs\PP_MonthlyStats_Global_PCA_Coverage_Sep2025.csv


,type,basemap_area,pa_area,pa_percent_coverage,oecm_area,oecm_percent_coverage,pa_oecm_area,pa_oecm_percent_coverage
0,terrestrial,1.345213e+08,22051643.0,16.39,1481782.0,1.10,23533425.0,17.49
1,eez,1.405167e+08,31710027.0,22.57,397459.0,0.28,32107486.0,22.85
2,abnj,2.224903e+08,3221699.0,1.45,462863.0,0.21,3684562.0,1.66
3,marine,3.630070e+08,34931726.0,9.62,860322.0,0.24,35792048.0,9.86
4,terrestrial+marine incl ATA,5.100660e+08,56983369.0,11.17,2342104.0,0.46,59325473.0,11.63
5,terrestrial+marine excl ATA,4.975282e+08,56983369.0,11.45,2342104.0,0.47,59325473.0,11.92
